Handling of images once they have been downloaded with planet_download.ipynb up to a product allowing us to train a model. 

Replaces the run_script_AF.py once created but not needed to re-run everything. 

Will apply the extraction from the zip file of downloaded images from Planet; Prepare the images creating the PNG images from the tif file allowing for the manual annotation; and will segment the images describe them for check, and cull them to suit the model requirements, and finally apply the model training. 

In [1]:
import sys
import os
import yaml

# Add the project root to sys.path (adjust as needed)
sys.path.append(os.path.abspath("counting_waterholes"))


import counting_boats.boat_utils.planet_utils as planet_utils
import counting_boats.boat_utils.testing as testing

c:\Users\fossatia\AppData\Local\miniconda3\envs\Boats\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
YOLOv5  v7.0-394-g86fd1ab2 Python-3.10.16 torch-1.12.1+cu113 CUDA:0 (GeForce GTX 1080, 8192MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Extraction of the file composite.tif obtained from the planet order and downloaded into the zip file. Renders a tif file and renames it with the date_aoi.tif outside the zip file. 

In [ ]:
import os
import yaml

import counting_boats.boat_utils.planet_utils

# Define the path to your zip file
zip_path = "images/raw_images"


# Run extraction
counting_boats.boat_utils.planet_utils.extract_zip(zip_path)

Prepare the raw tif image into a usable png in future steps. Creates a padded png image to exactly match the size dividable by the stride and tile size. 
Need to define the config to make sure it matches my paths and running the tif to png transformation. 

In [ ]:
import os
import yaml

import counting_boats.train

# #cfg config:
# with open("config_train_GPU.yaml", "r") as ymlfile:
#     cfg = yaml.load(ymlfile, Loader=yaml.FullLoader)
#     os.makedirs(cfg["output_dir"], exist_ok=True)
#     cfg["tif_dir"] = cfg.get(
#         "tif_dir", os.path.join(cfg["proj_root"], "images", "RawImages")
#     )  # This is generated so not included in the config file



#Run preparation of the tif files into png and renamed the tif. 
#prepare(r"C:\Users\adria\OneDrive - AdrianoFossati\Documents\MASTER Australia\RA\Waterholes_project\counting_waterholes\images\RawImages", cfg)
#Use relative paths not absolute 
counting_boats.train.prepare("config_train_GPU.yaml")
 

Once the png is created, as we are in the training of the model phase, I need to go on LabelMe (called here in the terminal) and manually annotate the waterholes which creates in the end a json file with all my bounding boxes. Will be needed now to segment the image and the corresponding labels. 

Once the manual annotation is done, run the segmentation of the created png image with padding.

In [ ]:
import os
import yaml

import counting_boats.train

#from counting_boats.train import segment


#segment the png images
counting_boats.train.segment("config_train_GPU.yaml", train_val_split=0.8)
# counting_boats.train.segment("config_test_Drive.yaml", train_val_split=0.8)


{'yolo_dir': 'C:/Users/fossatia/Documents/Waterholes_project/yolov5', 'python': 'python', 'path': 'D:/Waterholes_project/counting_waterholes/dummy_train', 'weights': 'C:/Users/fossatia/Documents/Waterholes_project/yolov5/runs/train/exp3/weights/best.pt', 'device': 'cuda:0', 'proj_root': 'C:/Users/fossatia/Documents/Waterholes_project/counting_waterholes/counting_waterholes', 'pngs': './pngs', 'raw_images': './raw_images', 'segmented_images': './segmented_images', 'labels': './labels', 'classifications': './classifications', 'plots': './plots', 'output_dir': 'D:/Waterholes_project/counting_waterholes/dummy_train', 'tif_dir': './images/RawImages', 'download_dir': './images/downloads', 'archive_dir': './archive_counting_waterholes', 'use_comet': False, 'workers': 6, 'BATCH_SIZE': 8, 'EPOCHS': 500, 'img_size': 416, 'img_stride': 104, 'TILE_SIZE': 416, 'STRIDE': 104, 'CONFIDENCE_THRESHOLD': 0.5, 'STAT_DISTANCE_CUTOFF_PIX': 50, 'STAT_DISTANCE_CUTOFF_LATLONG': 0.00025, 'COMPARE_DISTANCE_CUTOF

Saving Segments: 100%|██████████| 15933/15933 [01:18<00:00, 203.75it/s]


Skipped 13071 images
Empty 13857 images


After segmentation, we evaluate the results of the segmentation and production of material to train the model using the "train.describe" function. 
Run the bellow cell to describe from created paths of segmented images. 

In [ ]:
import sys
import os
import yaml

import counting_boats.train

#describe the created segmented images: 
counting_boats.train.describe("config_train_GPU.yaml")
# counting_boats.train.describe("config_test_Drive.yaml")


KeyError: 'train'

Apply the cull command which will remove images with no labels until 10% of the training set has no labels. This has to be done post segmentation as we don't know prior the the amount (depends on the segmentation). 
Using my created function as the official cull from Charlie doesn't work well. Mine runs all good. 

In [1]:
import sys
import os
import yaml
import random
import shutil
from pathlib import Path 

import counting_boats.train

#describe the created segmented images: 
# counting_boats.train.cull("config_train_GPU.yaml")
#AF: the cull function of Charlie seemed to run for ages... not sure why. So I develop an alternative one. 

counting_boats.train.cull_AF("config_train_GPU.yaml")


Analyzing label files in: results\labels\train
Looking for corresponding images in: results\images\train
Total label files found: 2069
Empty label files found: 787
Non-empty label files: 1282
Moving 645 empty label files to maintain 10% ratio

--- SUMMARY ---
Total label files moved: 645
Total image files moved: 645
Remaining total label files: 1424
Remaining empty label files: 142
Empty labels now make up 9.97% of the dataset
Empty labels moved to: results\labels\moved_empty_labels
Corresponding images moved to: results\images\moved_empty_images

SUCCESS: Empty labels now make up 10% or less of the dataset.


Once the cull function is applied reducing the no instance images amount to 10%, we can try to train the model. 
But first let's reorganise the folders to be used in the model. 

In [1]:
import sys
import os
import yaml

import counting_boats.train

counting_boats.train.reorganize_folders("config_train_GPU.yaml")

Copying from results\images\val to training\val\images
Successfully copied to training\val\images
Copying from results\images\train to training\train\images
Successfully copied to training\train\images
Copying from results\labels\val to training\val\labels
Successfully copied to training\val\labels
Copying from results\labels\train to training\train\labels
Successfully copied to training\train\labels

----- REORGANIZATION SUMMARY -----
Successful operations: 4
Failed operations: 0

All folders were successfully reorganized!

New structure:
- training/val/images (copied from results/images/val)
- training/val/labels (copied from results/labels/val)
- training/train/images (copied from results/images/train)
- training/train/labels (copied from results/labels/train)


Modifiy now in the config file of the yolo model training the path directory of the run. then run the following code and go in the yolov5 folder to run it and train the model. 

This code provides you with the command to excecute in the cmd of the yolo.  

In [2]:
import sys
import os
import yaml

import counting_boats.train

#describe the created segmented images: 
counting_boats.train.train("config_train_GPU.yaml")

python C:/Users/fossatia/Documents/Waterholes_project/yolov5/train.py --device cuda:0 --img 416 --batch 8 --workers 6 --epochs 500 --data config_train_GPU.yaml --weights ./data/NN_weights.pt --save-period 50


Training of the model when ordered directly into the cmd panel and not via notebook: 
Need to figure out how to do it from here to streamline the whole process though...

In [ ]:
python train.py --workers 2 --img 416 --batch 8 --epochs 150 --data config_train_GPU_yolo.yaml --weights yolov5s.pt --cache disk

End of this script. 